# Tutorial: Compare NEON and EMIT data for SOAP site

## Second of two notebooks

### Authors: Hannah Rieder, Randi Neff, Bridget Hass

#### last updated: 8/7/25

This tutorial is intended for Earth Science data professionals. Additional details and an overall summary of this project will be available on the [Earth Lab Blog](https://earthlab.colorado.edu/earth-data-analytics-professional-graduate-certificate/earth-data-analytics-certificate-cohorts). In this tutorial, we will learn how to compare resolution of two surface reflectance datasets by calculating Canopy Water Content (CWC) over individual tiles at the Soaproot Saddle (SOAP) field site in the Sierra National Forest in California. We will also evaluate forest health. One of the tiles we will compare was within the fire perimeter of the Creek Fire in fall 2020; the other tile will be adjacent to the burned one but was not burned by the Creek Fire. The hyperspectral data for the CWC calculation comes from the [National Ecological Observatory Network's (NEON) Level 3 Spectrometer orthorectified surface bidirectional reflectance - mosaic data product](https://data.neonscience.org/data-products/DP3.30006.002) and the [Earth Surface Mineral Dust Source Investigation (EMIT) L2A Reflectance Data Product](https://www.earthdata.nasa.gov/data/catalog/lpcloud-emitl2arfl-001).

## The objectives of this tutorial (divided between two notebooks) are to:

* Use co-located data from NEON and EMIT
* Calculate Canopy Water Content (CWC) from NEON and EMIT hyperspectral data
* Evaluate CWC data at different scales
* Compare between burned and unburned areas

DATA The data provided with this tutorial were derived from existing code at:

* NEON Spectrometer orthorectified surface bidirectional reflectance data.
* Shapefiles for NEON burned and unburned tiles which are found in the DATA folder.
* EMIT L2A Estimated Surface Reflectance granule(s) that cover the NEON burned and unburned tiles.
* [Land Processes Distributed Active Archive Center (LP DAAC)](https://nasa.github.io/VITALS/python/03_EMIT_CWC_from_Reflectance.html#cwc-of-a-single-point).

Additional data will be downloaded programmatically within this tutorial.

## What we should have after completing notebook 1:

* two EMIT cropped datasets (one for the burned tile and one for the unburned tile) exported to netcdf files that are saved to the reflectance directory at `../data/refl/`
* two NEON reflectance datasets (one for the burned tile and one for the unburned tile). These datasets will already have been converted from hdf5 format into xarray, have the scale factor applied, have bad bands set to NaN, and be exported to netcdf files that are saved to the reflectance directory at `../data/refl/`**

## Tutorial Outline for Notebook 2 - Canopy Water Content Comparison

0. Import Standard Packages
1. Setup

   1.1 Create Data and Scripts (modules) Directories

   1.2 Download and Import Necessary Scripts

   1.3 Download and Open the Refractive Index of Liquid Water per Wavelength CSV
3. Open NEON and EMIT Reflectance Data
4. Calculate Canopy Water Content (CWC)
5. Compare CWC Datasets

## Acknowledgements:

The existing [3 Equivalent Water Thickness/Canopy Water Content from Imaging Spectroscopy Data](https://nasa.github.io/VITALS/python/03_EMIT_CWC_from_Reflectance.html) tutorial in the [NASA VITALS Repository](https://github.com/nasa/VITALS). That existing notebook is a tutorial for how to calculate Equivalent Water Thickness (EWT) or Canopy Water Content (CWC) from EMIT L2A reflectance data over a nature preserve near Santa Barbara, California. We followed that tutorial with EMIT L2A reflectance data over the the SOAP field site while exploring CWC and used it to help create this notebook to calculate CWC over the SOAP site and using NEON reflectance data instead of EMIT. Specifically, section "3.5 CWC Calculation of an ROI" of the NASA VITALS existing tutorial was used to calculate CWC below.

Additionally, in section 3 below in the `calc_open_cwc()` function, there are two functions: `calc_ewt()` and `calc_ewt_neon()`. These functions were created by modifying an [existing calc_ewt() function in the NASA VITALS repo](https://github.com/nasa/VITALS/blob/main/python/modules/ewt_calc.py). The original calc_ewt() function was modified so that it would work with a .nc file of NEON surface reflectance data. Specifically, NEON surface reflectance data does not have the metadata that EMIT surface reflectance data does and for the original calc_ewt() function to work, it requires that metadata be part of the .nc surface reflectance file.

### 0. Import Standard Packages

In [ ]:
# Import Packages
import os, sys # Python module to create and acces file paths
import pathlib
# Some cells may generate warnings that we can ignore.
# Comment below lines to see.
import warnings
warnings.filterwarnings('ignore')

import numpy as np # Work with multi-dimensional arrays
import xarray as xr # Work with labelled multi-dimenstional arrays
from osgeo import gdal # Work with raster and vector geospatial data
import rasterio as rio # Work with geospatial raster data
import rioxarray as rxr # Work with raster arrays
from matplotlib import pyplot as plt # Plotting data
import hvplot.xarray # Plot multi-dimensional arrays
import hvplot.pandas # Plot DataFrames/Series
import pandas as pd # Work with DataFrames
import geopandas as gpd # Work with geospatial shapefiles
import earthaccess # Search for, download, & stream NASA earth data
from tqdm.notebook import tqdm # Progress bars on loops
import requests
from scipy import stats
from scipy.optimize import least_squares # Nonlinear least-squares
import holoviews as hv

import h5py # Work with NEON reflectance data

### 1. Setup

In the setup section, we will do three things:
1. create directories to store the input and output data and scripts for this project,
2. download and import the extra necessary scripts needed for this tutorial, and
3. download a comma separated value (CSV) file needed for the CWC calculation function.

### 1.1 Create Data and Scripts (modules) Directories

#### *If you completed Tutorial Notebook 01, you will already have the data and scripts directories created. It is recommended to still run the code in 1.1 to verify that the directories have been made and to create the canopy water content data directory.*

The directories we will make are: an overarching data directory, a reflectance data directory, a CWC data directory, a shapefiles data directory, and a modules directory.

The overarching data directory will contain the reflectance and CWC data directories and a CSV file containing lab measurements of the complex refractive index of liquid water. 

The reflectance data directory should contain the two EMIT cropped datasets (one for the burned tile and one for the unburned tile) and the two NEON reflectance datasets (one for the burned tile and one for the unburned tile) created in Tutorial Notebook 01. All of these datasets should be NetCDF files.

The shapefiles data directory will contain shapefiles for the SOAP site and the tiles.

The CWC data directory will be where we store the results of this tutorial notebook: the CWC calculations of the burned and unburned tiles calculated using the cropped EMIT and NEON reflectance data.

The modules directory must be in the same folder as where you have these tutorial notebooks stored for some of the imported functions to work. In this modules directory, we will manually download some Python scripts (.py files). The scripts contain various functions we'll use in this tutorial.

The CWC calculation functions (calc_ewt and calc_ewt_neon) are expecting the data and the k_liquid_water_ice.csv file to be stored in a directory that is at the same file level as where you have this notebook stored. In the cell below, the `data_dir = r"../data"` code ensures that the data directory will be at the same file level as where this tutorial notebook is stored.

Here is a visual of how the directory and file structure will look once these directories are created and all files are downloaded and created from tutorial notebooks 01 and 02:

```
project-root/
│
├── data/                     # Main folder for data
│   ├── cwc/                  # Subfolder for canopy water content (CWC) data genearted in tutorial_notebook_02
│   │   ├── emit_burn_cwc.nc               
│   │   ├── emit_soap_burned_cwc.tif               
│   │   ├── emit_soap_unburned_cwc.tif
│   │   ├── emit_unburn_cwc.nc                           
│   │   ├── neon_burn_cwc.nc               
│   │   ├── neon_burn_refl_cwc.tif               
│   │   ├── neon_unburn_cwc.nc               
│   │   └── neon_unburn_refl_cwc.tif          
│   │
│   ├── refl/                 # Subfolder for reflectance data generated in tutorial_notebook_01
│   │   ├── EMIT_L2A_RFL_001_20230731T205320_2321214_004.nc
│   │   ├── EMIT_L2A_RFL_20230731_SOAP.nc
│   │   ├── emit_soap_burned.nc
|   |   ├── emit_soap_unburned.nc
|   |   ├── neon_burn_refl.nc
|   |   ├── NEON_D17_SOAP_DP3_298000_4100000_bidirectional_reflectance.h5
|   |   ├── NEON_D17_SOAP_DP3_298000_4101000_bidirectional_reflectance.h5
|   |   └── neon_unburn_refl.nc
|   ├── shapefiles/           # Subfolder for shapefiles from the AOP-EMIT GiutHub repo & generated in tutorial_notebook_01
|   |   ├── AOPflightBoxes folder 
|   |   ├── AOPflightBoxes_0 folder
|   |   ├── NEON_D17_SOAP_DPQA_298000_4100000_boundary.dbf
|   |   ├── NEON_D17_SOAP_DPQA_298000_4100000_boundary.prj
|   |   ├── NEON_D17_SOAP_DPQA_298000_4100000_boundary.shp
|   |   ├── NEON_D17_SOAP_DPQA_298000_4100000_boundary.shx
|   |   ├── NEON_D17_SOAP_DPQA_298000_4101000_boundary.dbf
|   |   ├── NEON_D17_SOAP_DPQA_298000_4101000_boundary.prj
|   |   ├── NEON_D17_SOAP_DPQA_298000_4101000_boundary.shp
|   |   └── NEON_D17_SOAP_DPQA_298000_4101000_boundary.shx
|   └── k_liquid_water_ice.csv
│
└── notebooks/                # Subfolder for modules and tutorial notebooks
    ├── modules/              # Subfolder for Python scripts for processing and analysis
    |   ├── __pycache__
    |   ├── emit_tools.py
    |   ├── ewt_calc2.py
    |   └── test_functions.py
    ├── 01_NEON_EMIT_tutorial_notebook.ipynb
    └── 02_NEON_EMIT_tutorial_notebook.ipynb
```

In [ ]:
# Define the file path for the data directory
data_dir = r"../data"

# Create/check for the data_dir
if not os.path.exists(data_dir):
    os.makedirs(data_dir)
    print(f'data directory made here: {data_dir}')
else:
    print(f'data directory already exists here: {data_dir}')

In [ ]:
# List of directories names to make
dir_list = ["refl", "cwc"]

# Define the root path where the directories will be created
root_path = data_dir

# Use a for loop to create/check for the directories in the dir_list
for dir_name in dir_list:
    full_path = os.path.join(root_path, dir_name)
    if not os.path.exists(full_path):
        os.makedirs(full_path)
        print(f'directory made here: {full_path}')
    else:
        print(f'directory already exists here: {full_path}')

In [ ]:
# Define the file path for the modules directory
modules_dir = r"./modules"

# Create/check for the modules_dir
if not os.path.exists(modules_dir):
    os.makedirs(modules_dir)
    print(f'modules directory made here: {modules_dir}')
else:
    print(f'modules directory already exists here: {modules_dir}')

### 1.2 Download and Import Necessary Scripts

#### *If you completed Tutorial Notebook 01, you will already have the test_functions.py and emit_tools.py modules downloaded. However, you still need to follow the instructions below to download and save the ewt_calc2.py module. Also, make sure to still run the code in the cell below to import necessary functions from those modules.*

The Python modules that we will put in the modules directory (ewt_calc2.py, emit_tools.py, and test_functions.py) contain functions that will allow us to calculate and visualize CWC and see a directory's contents. The emit_tools.py script is required for the functions in the ewt_calc2.py script to work. We can access and download the scripts with the following steps:

1. Follow [this link](https://github.com/NEONScience/AOP-EMIT/blob/main/notebooks/modules/ewt_calc2.py) to access the ewt_calc2.py module.
2. Manually download the raw file to your computer.
3. Move the ewt_calc2.py file into the modules directory we made above (`modules_dir = r"./modules"`). **It is important that the ewt_calc2.py script is in the modules_dir; the import code below expects the script to be in the modules_dir.**
4. Run the code in the cell below to import functions from the ewt_calc2.py module into this notebook.
5. Repeat steps 1-4 except, for step 1, follow [this link](https://github.com/NEONScience/AOP-EMIT/blob/main/notebooks/modules/test_functions.py) to access the test_functions.py module.
6. Repeat steps 1-4 except, for step 1, follow [this link](https://github.com/NEONScience/AOP-EMIT/blob/main/notebooks/modules/emit_tools.py) to access the emit_tools.py module.

In [ ]:
# Import functions from the python scripts in the modules directory
from modules.test_functions import data_download_tracker, surfrfl_hvplot_image
from modules.emit_tools import emit_xarray # Open EMIT datasets as xarray.Dataset
from modules.ewt_calc2 import calc_ewt, calc_ewt_neon # Canopy water content fxn

### 1.3 Download and Open the Refractive Index of Liquid Water per Wavelength CSV

The CWC calculation functions (calc_ewt and calc_ewt_neon) requires this CSV file to work. The name of the CSV file is k_liquid_water_ice.csv, which can be found in the EMIT VITALS GitHub repository data folder. We can access and download the k_liquid_water_ice.csv file with the following steps:

1. Follow [this link](https://github.com/nasa/VITALS/blob/main/data/k_liquid_water_ice.csv) to access the k_liquid_water_ice.csv file in the EMIT VITALS repository.
2. Manually download the raw file to your computer.
3. Move the k_liquid_water_ice.csv file into the data directory we made above (`data_dir = r"../data"`). **It is important that the k_liquid_water_ice.csv file is in the data_dir; the calc_ewt and calc_ewt_neon functions expect the CSV file to be in the data_dir.**
4. Run the code in the cell below to open and look at the k_liquid_water_ice.csv file in this notebook. 

The existing [EMIT VITALS CWC tutorial notebook](https://nasa.github.io/VITALS/python/03_EMIT_CWC_from_Reflectance.html#setup) has some helpful details about what this k_liquid_water_ice.csv file is:
>We need some lab measurements of the complex refractive index of liquid water to obtain the wavelength-dependent absorption coefficients. They are calculated by taking four times the product of Pi and the imaginary part of the refractive index, divided by wavelength. The refractive index of liquid water per wavelength is provided by the k_liquid_water_ice.csv in the data folder.

In [ ]:
# Define file path to k_liquid_water_ice.csv file
wp_fp = ("../data/k_liquid_water_ice.csv")

# Read k_liquid_water_ice.csv file into a DataFrame
k_wi = pd.read_csv(wp_fp)

# Check k_wi DataFrame
k_wi.head()

### 2. Open NEON and EMIT Reflectance Data

In Tutorial Notebook 01, we downloaded and processed EMIT L2A Reflectance data and NEON Level 3 Spectrometer orthorectified surface bidirectional reflectance - mosaic data. Specifically, we downloaded EMIT reflectance data for one granule from July 31, 2023 and cropped it to the burned and unburned tiles of interest. We downloaded NEON bidirectional reflectance from 2024 for the burned and unburned tiles. We processed the NEON reflectance data in the following ways:

1. scaling the reflectance data by the scale factor (NEON data are saved in an integer format, scaled by 10000, in order to save on space)
2. setting the water vapor absorption windows (defined as "bad band windows") to NaN. Similar to the EMIT data, "good_wavelengths" are provided as one of the Coordinates in the neon_burn_refl_ds xarray dataset, so we can use that information to keep only the valid wavelengths
3. writing the CRS (coordinate reference system information)

We also exported the cropped and processed EMIT and NEON reflectance data to NetCDF files. These NetCDF files were saved in the `../data/refl` directory.

In the code cells below, we are going to define file paths to the cropped and processed EMIT and NEON reflectance. We'll need the file path variables (`neon_burn_fp`, `emit_burn_fp`, etc) for the CWC calculation functions below.

*Note: to define the file path for and open the EMIT and NEON reflectance data, we are using the same code cell 4 times below. Normally, this is repetitive and we would create a function or loop to do this more efficiently. However, since we are just loading in and looking at the datasets and it is not the main focus of this tutorial, we will not focus on making it more efficient.*

In [ ]:
# Check to see that the NetCDF files are in the ../data/refl directory
data_download_tracker(
    # Path to the data_dir
    absolute_soap_path = r'../data/',
    # Folder name in the data_dir that we want to explore
    folder_names = ['refl'],
    # File type we're looking for
    extension = '.nc'
)

In [ ]:
# Define file path to the burned NEON reflectance NetCDF file
neon_burn_refl_fp = "../data/refl/neon_burn_refl.nc"

# Open NetCDF NEON burned dataset
neon_burn_refl_ds = xr.open_dataset(neon_burn_refl_fp, decode_coords="all")

# Optionally, uncomment to view neon_burn_refl_ds
# neon_burn_refl_ds

In [ ]:
# Define file paths to the unburned NEON reflectance NetCDF file
neon_unburn_refl_fp = "../data/refl/neon_unburn_refl.nc"

# Open NetCDF NEON unburned dataset
neon_unburn_refl_ds = xr.open_dataset(neon_unburn_refl_fp, decode_coords="all")

# Optionally, uncomment to view neon_unburn_refl_ds
#neon_unburn_refl_ds

In [ ]:
# Define file path to the cropped burned EMIT reflectance NetCDF file
emit_burn_refl_fp = "../data/refl/emit_soap_burned.nc"

# Open NetCDF EMIT burned dataset
emit_burn_refl_ds = xr.open_dataset(emit_burn_refl_fp, decode_coords="all")

# Optionally, uncomment to view emit_burn_refl_ds
#emit_burn_refl_ds

In [ ]:
# Define file path to the cropped unburned EMIT reflectance NetCDF file
emit_unburn_refl_fp = "../data/refl/emit_soap_unburned.nc"

# Open NetCDF EMIT unburned dataset
emit_unburn_refl_ds = xr.open_dataset(emit_unburn_refl_fp, decode_coords="all")

# Optionally, uncomment to view emit_unburn_refl_ds
#emit_unburn_refl_ds

### 3. Calculate and Visualize Canopy Water Content (CWC)

We will calculate CWC using the processed NEON reflectance data and the cropped EMIT reflectance data. As described in the [NASA VITALS Equivalent Water Thickness/Canopy Water Content from Imaging Spectroscopy Data tutorial](https://nasa.github.io/VITALS/python/03_EMIT_CWC_from_Reflectance.html): 
> CWC correlates with vegetation type and health, as well as wildfire risk. The methods used here to calculate CWC are based on the [ISOFIT python package](https://github.com/isofit/isofit/tree/main). The Beer-Lambert physical model used to calculate CWC is described in [Green et al. (2006)](https://agupubs.onlinelibrary.wiley.com/doi/10.1029/2005WR004509) and [Bohn et al. (2020)](https://www.sciencedirect.com/science/article/abs/pii/S0034425720300778?via%3Dihub). It uses wavelength-dependent absorption coefficients of liquid water to determine the absorption path length as a function of absorption feature depth. Of note, this model does not account for multiple scattering effects within the canopy and may result in overestimation of CWC (Bohn et al., 2020).

For our specific purposes, we're calculating CWC to see the difference in resolution between the NEON and EMIT reflectance data. Later on, we will also compare the CWC values and apply our findings to forest health.

We're going to start by learning about the `calc_ewt()` and `calc_ewt_neon()` functions imported in the beginning:

In [ ]:
# Learn about calc_ewt function
help(calc_ewt)

In [ ]:
# Learn about neon_calc_ewt function
help(calc_ewt_neon)

Before calculating CWC, we have to decide where we want the results to be stored, which is called the `out_dir`. Our `out_dir` will be the `../data/cwc/` directory we made in the Setup section of this notebook.

In [ ]:
# Set output directory where results of CWC function will be stored
out_dir = r"../data/cwc/"

Depending on the dataset and your computer, calculating CWC can take up to 3 hours. Because of this, let's create a function with conditional statements so that if CWC has already been calculated and saved to a NetCDF file, CWC won't be calculated again.

In [ ]:
def calc_open_cwc(
    dsource_burntype_refl_fp,
    dsource,
    burntype,
    out_dir,
    **kwargs
):
    """
    Calculate or display canopy water content of a surface reflectance dataset.

    Use existing calc_ewt() or calc_ewt_neon() function to calculate canopy
    water content (CWC) of an EMIT or NEON surface reflectance dataset. An
    xarray.Dataset containing the CWC values will be created,
    `dsource_burntype_cwc_ds`. The `dsource_burntype_cwc_ds` xarray.Dataset
    will be exported to a NetCDF file, which will be stored in the `out_dir`.
    Additionally, a cloud-optimized geotiff (COG) file will be created with
    the CWC values and stored in the `out_dir`.

    Parameters
    ----------
    dsource_burntype_refl_fp : str
        File path to the EMIT or NEON NetCDF (.nc) surface reflectance dataset

    dsource : {r"emit", r"neon"}
        Source of the surface reflectance dataset, either EMIT or NEON

    burntype : {r"burn", r"unburn"}
        Whether the surface reflectance dataset is from a burned or unburned 
        geographic area

    out_dir : str
        File path to where the results of the function will be stored (NetCDF
        version of the `dsource_burntype_cwc_ds`, geotiff)

    ewt_detection_limit : float
        Upper detection limit for CWC

    **kwargs
        Extra arguments to calc_ewt() and calc_ewt_neon() functions
    
    Returns
    -------
    dsource_burntype_cwc_ds : xarray.Dataset
        An xarray.Dataset of CWC values

    Notes
    -----
    The existing functions in this `calc_open_cwc()` function, calc_ewt() and
    calc_ewt_neon(), are modified from an existing calc_ewt() function in the
    NASA VITALS repository[1]_. The existing calc_ewt() function was modified
    to work with NEON reflectance data.

    References
    ----------
    .. [1] VITALS/python/modules/ewt_calc.py at main · nasa/VITALS. (n.d.).
    Retrieved August 5, 2025, from
    https://github.com/nasa/VITALS/blob/main/python/modules/ewt_calc.py
    """
    # Define file path to .nc reflectance file
    dsource_burntype_refl_fp = dsource_burntype_refl_fp
    print(f'this is the dsource_burntype_refl_fp: {dsource_burntype_refl_fp}')
    # Define file path for CWC .nc file
    dsource_burntype_cwc_fp = f"{out_dir}{dsource}_{burntype}_cwc.nc"
    print(f'this is the dsource_burntype_cwc_fp: {dsource_burntype_cwc_fp}')
    
    # If CWC has already been calculated and saved to a .nc file,
    if os.path.exists(dsource_burntype_cwc_fp):
        print('CWC path exists, displaying CWC dataset...')
        # Open the CWC .nc file,
        dsource_burntype_cwc_ds = xr.open_dataset(
            dsource_burntype_cwc_fp, decode_coords="all")
        # Display the CWC dataset, and
        display(dsource_burntype_cwc_ds)
        # Print the file path to the CWC dataset .nc file
        print(f'Here is where the CWC dataset NetCDF file is saved: '
              f'{dsource_burntype_cwc_fp}')
    
    # If CWC has not been calculated and saved to a .nc file,
    else:
        # Calculate CWC for EMIT surface reflectance data
        if 'emit' in dsource_burntype_refl_fp:
            print(f'{dsource_burntype_cwc_fp} does not exist,'
                  f' calculating CWC for {dsource_burntype_refl_fp}...')
            dsource_burntype_cwc_ds = calc_ewt(
                # File path to .nc file of reflectance dataset 
                dsource_burntype_refl_fp,
                # File path to where results will be stored
                out_dir,
                # Define upper detection limit for CWC
                ewt_detection_limit=1.5,
                # Have calc_ewt function return the geotiff
                return_cwc=True
            )
            print(f'CWC for {dsource_burntype_refl_fp} calculated,'
                  f' here is the CWC dataset:')
            display(dsource_burntype_cwc_ds)
            print('\n')
            print('Now exporting CWC dataset to a NetCDF file...')
            # Export CWC dataset to a .nc file and save it in the `out_dir`
            dsource_burntype_cwc_ds.to_netcdf(
                f"{out_dir}{dsource}_{burntype}_cwc.nc")
            print(f'NetCDF file created here:'
                  f' {out_dir}{dsource}_{burntype}_cwc.nc')
        
        # Calculate CWC for NEON surface reflectance data
        if 'neon' in dsource_burntype_refl_fp:
            print(f'{dsource_burntype_cwc_fp} does not exist,'
                  f' calculating CWC for {dsource_burntype_refl_fp}...')
            dsource_burntype_cwc_ds = calc_ewt_neon(
                # File path to .nc file of reflectance dataset
                dsource_burntype_refl_fp,
                # File path to where results will be stored
                out_dir,
                # Define upper detection limit for CWC
                ewt_detection_limit=1.5,
                # Have calc_ewt function return the geotiff
                return_cwc=True
            )
            print(f'CWC for {dsource_burntype_refl_fp} calculated,'
                  f' here is the CWC dataset:')
            display(dsource_burntype_cwc_ds)
            print('\n')
            print('Now exporting CWC dataset to a NetCDF file...')
            # Export CWC dataset to a .nc file and save it in the `out_dir`
            dsource_burntype_cwc_ds.to_netcdf(
                f"{out_dir}{dsource}_{burntype}_cwc.nc")
            print(f'NetCDF file created here:'
                  f' {out_dir}{dsource}_{burntype}_cwc.nc')
            
    return dsource_burntype_cwc_ds

#### 3.1.1 Calculate CWC for the EMIT surface reflectance data

Calculating CWC for the EMIT reflectance data for the first time is much shorter than calculating CWC for NEON reflectance Data. The EMIT calculation should take about a minute max.



In [ ]:
%%time
emit_burn_cwc_ds = calc_open_cwc(
    # File path to .nc file of EMIT burned reflectance dataset
    emit_burn_refl_fp,
    # Data source is NEON
    dsource = r"emit",
    # Burntype is burned
    burntype = r"burn",
    # File path to out directory where CWC COG and .nc file will be stored
    out_dir = r"../data/cwc/"
)


In [ ]:
%%time
emit_unburn_cwc_ds = calc_open_cwc(
    emit_unburn_refl_fp,
    dsource = r"emit",
    burntype = r"unburn",
    out_dir = r"../data/cwc/"
)

#### 3.1.2 Calculate CWC for the NEON surface reflectance data

**NOTE:** Due to the high resolution of the NEON reflectance data, calculating NEON CWC for the first time can take 45 minutes to 3 hours, depending on your local machine.

**When you calculate NEON CWC, make sure your computer is charging and won't fall asleep.** 

In [ ]:
%%time
neon_burn_cwc_ds = calc_open_cwc(
    neon_burn_refl_fp,
    dsource = r"neon",
    burntype = r"burn",
    out_dir = r"../data/cwc/"
)

In [ ]:
%%time
neon_unburn_cwc_ds = calc_open_cwc(
    neon_unburn_refl_fp,
    dsource = r"neon",
    burntype = r"unburn",
    out_dir = r"../data/cwc/"
)

### 3.2 Visualize CWC Datasets

In [ ]:
emit_neon_burn_unburn_cwc_plot = hv.Layout(
    # Plot emit_unburned_cwc_ds
    surfrfl_hvplot_image(
        emit_unburn_cwc_ds,
        # Set plot title
        plottitle=(f"EMIT CWC ({emit_unburn_cwc_ds.cwc.units})"
                   f" July 2023 - SOAP Unburned Tile"),
        # Set colormap
        cmap='jet_r',
        # Set colorbar label and limit
        clabel="Canopy Water Content (g/cm^2)",
        clim = (0,0.5),
        # Set 
        x='longitude', y='latitude',
        # Set frame size and font scale
        frame_width=360,
        frame_height=202,
        fontscale=1
    )
    +
    # Plot neon_unburned_cwc_ds
    surfrfl_hvplot_image(
        neon_unburn_cwc_ds,
        # Set plot title
        plottitle=(f"NEON CWC ({neon_unburn_cwc_ds.cwc.units})"
                   f" June 2024 - SOAP Unburned Tile"),
        # Set colormap
        cmap='jet_r',
        # Set colorbar label and limit
        clabel="Canopy Water Content (g/cm^2)",
        clim = (0,0.5),
        x='x', y='y',
        # Set frame size and font scale
        frame_width=360,
        frame_height=202,
        fontscale=1
    )    
    +
    # Plot emit_burn_cwc_cog_ds
    surfrfl_hvplot_image(
        emit_burn_cwc_ds,
        # Set plot title
        plottitle=(f"EMIT CWC ({emit_burn_cwc_ds.cwc.units})"
                   f" July 2023 - SOAP Burned Tile"),
        # Set colormap
        cmap='jet_r',
        # Set colorbar label and limit
        clabel="Canopy Water Content (g/cm^2)",
        clim = (0,0.5),
        x='longitude', y='latitude',
        # Set frame size and font scale
        frame_width=360,
        frame_height=202,
        fontscale=1
    )
    +
    # Plot neon_burn_cwc_ds
    surfrfl_hvplot_image(
        neon_burn_cwc_ds,
        # Set plot title
        plottitle=(f"NEON CWC ({neon_burn_cwc_ds.cwc.units})"
                   f" June 2024 - SOAP Burned Tile"),
        # Set colormap
        cmap='jet_r',
        # Set colorbar label and limit
        clabel="Canopy Water Content (g/cm^2)",
        clim = (0,0.5),
        x='x', y='y',
        # Set frame size and font scale
        frame_width=360,
        frame_height=202,
        fontscale=1
    )
).cols(2)

# Show plot
emit_neon_burn_unburn_cwc_plot

#### Canopy water content plots show the contrast in resolution between EMIT and NEON reflectance data. EMIT reflectance data and CWC measurements could be used to summarize a forest, however for true details, NEON reflectance data and CWC provide a more accurate picture.

The CWC calculations provide great opportunities to compare the resolution of EMIT and NEON reflectance data as well as burned and unburned tiles. We can clearly see that the EMIT CWC plots for the burned and unburned tiles have fewer pixels and less details than the NEON CWC plots. This is because the EMIT reflectance data has a lower resolution than the NEON reflectance data.

However, despite the difference in resolution, we can see that the low resolution data from space (EMIT) agrees with the high resolution data from an aerial survey (NEON). Looking at just the SOAP unburned tile CWC plots, there are clear areas where both plots show higher and lower CWC. For example, in the southwest corner of the unburned tile, both EMIT and NEON CWC plots show low CWC. In the south east quadrant of the unburned tile, both EMIT and NEON CWC plots show higher CWC. Looking at the SOAP burned tile CWC plots, we can see dark patches of low CWC in the middle of the top half of the tile in both the EMIT and NEON CWC plots.

In terms of evaluating forest health, we can conclude that EMIT reflectance data and CWC measurements can summarize a forest's health, however it will likely miss the nuance that NEON data can provide. The EMIT CWC calculations are lower overall than the NEON CWC calculations. The highest values for CWC with EMIT data show up as green (0.3 g/cm^2), whereas the highest NEON CWC calculations show up as blue (0.5 g/cm^2). The EMIT CWC plots show more pixels in dark orange and red than the NEON CWC plots do. This could be due to a few different things. First, EMIT CWC is based on 2023 reflectance measurements while NEON CWC is based on 2024 reflectance measurements; the forest will have continued to recover from the Creek Fire between Summer 2023 and Summer 2024. Also, EMIT CWC could be lower because of resolution differences, or other factors such as different atmospheric corrections applied to the two datasets.

### 4. Compare CWC Datasets

To quantitatively compare the CWC datasets, we are going to resample the NEON data and find the difference between the resampled NEON datasets and the EMIT datasets.

#### First, we are going to read in the cloud-optimized geotiff (COG) files generated by the `calc_open_cwc()` function:


In [ ]:
# Define file paths to cloud-optimized geotiff files generated by calc_open_cwc()
emit_burn_cwc_cog = "../data/cwc/emit_soap_burned_cwc.tif"
emit_unburn_cwc_cog = "../data/cwc/emit_soap_unburned_cwc.tif"

# Read in the EMIT Burned and Unburned CWC COGs
emit_burn_cwc_cog_ds = rxr.open_rasterio(
    emit_burn_cwc_cog,
    band_as_variable=True)
emit_burn_cwc_cog_ds = emit_burn_cwc_cog_ds.rename_vars({"band_1": "cwc"})

emit_unburn_cwc_cog_ds = rxr.open_rasterio(
    emit_unburn_cwc_cog,
    band_as_variable=True)
emit_unburn_cwc_cog_ds = emit_unburn_cwc_cog_ds.rename_vars({"band_1": "cwc"})

In [ ]:
# Define file paths to cloud-optimized geotiff files generated by calc_open_cwc()
neon_burn_cwc_cog = "../data/cwc/neon_burn_refl_cwc.tif"
neon_unburn_cwc_cog = "../data/cwc/neon_unburn_refl_cwc.tif"

# Read in the NEON Burned and Unburned CWC COGs
neon_burn_cwc_cog_ds = rxr.open_rasterio(
    neon_burn_cwc_cog,
    band_as_variable=True)
neon_burn_cwc_cog_ds = neon_burn_cwc_cog_ds.rename_vars({"band_1": "cwc"})

neon_unburn_cwc_cog_ds = rxr.open_rasterio(
    neon_unburn_cwc_cog,
    band_as_variable=True)
neon_unburn_cwc_cog_ds = neon_unburn_cwc_cog_ds.rename_vars({"band_1": "cwc"})

#### Second, we will spatially resample the NEON data to match the EMIT data so that the CWC datasets have the same resolution. After resampling, we'll subtract the resampled NEON dataset from the the EMIT dataset for the burned and unburned tiles:

In [ ]:
# Spatially resample NEON data to match EMIT data
neon_burn_resampled_ds = (
    neon_burn_cwc_cog_ds.rio.reproject_match(emit_burn_cwc_cog_ds)
)

# Calculate difference between emit_burn_cwc_cog_ds and neon_burn_resampled_ds
burned_cwc_difference_ds = emit_burn_cwc_cog_ds - neon_burn_resampled_ds

# View burned_cwc_difference
burned_cwc_difference_ds

In [ ]:
# Spatially resample NEON data to match EMIT data
neon_unburn_resampled_ds = (
    neon_unburn_cwc_cog_ds.rio.reproject_match(emit_unburn_cwc_cog_ds)
)

# Calculate difference between emit_unburn_cwc_ds and neon_unburn_resampled_ds
unburned_cwc_difference_ds = emit_unburn_cwc_cog_ds - neon_unburn_resampled_ds

# View unburned_cwc_difference
unburned_cwc_difference_ds

#### Third, we're going to process the difference datasets by replacing the 'nodata' outlier value with NaN:

In [ ]:
# Create a copy of the data to avoid modifying the original
clean_burn_cwc_difference = burned_cwc_difference_ds['cwc'].copy()

# Replace the 'nodata' outlier value with NaN
# We'll use a threshold (e.g., > 1000) to safely capture the outlier
clean_burn_cwc_difference = clean_burn_cwc_difference.where(
    clean_burn_cwc_difference < 1000, np.nan)

In [ ]:
# Create a copy of the data to avoid modifying the original
clean_unburn_cwc_difference = unburned_cwc_difference_ds['cwc'].copy()

# Replace the 'nodata' outlier value with NaN
# We'll use a threshold (e.g., > 1000) to safely capture the outlier
clean_unburn_cwc_difference = clean_unburn_cwc_difference.where(
    clean_unburn_cwc_difference < 1000, np.nan)


#### Fourth, we're going to find the mean of the processed burned and unburned CWC difference datasets:

In [ ]:
# Calculate the mean of the cleaned difference data
# The .values[()] extracts the numerical value from the xarray object
mean_unburn_difference = clean_unburn_cwc_difference.mean().values[()]

# Print the result
print(f"The mean unburned Canopy Water Content (CWC) difference is:"
      f" {mean_unburn_difference:.4f} g/cm²")

# Calculate the mean of the cleaned difference data
mean_burn_difference = clean_burn_cwc_difference.mean().values[()]

# Print the result
print(f"The mean burned Canopy Water Content (CWC) difference is:"
      f" {mean_burn_difference:.4f} g/cm²")

#### Last, we're going to plot the processed difference datasets:

In [ ]:
# Create a figure with two subplots (side by side)
fig, axes = plt.subplots(1, 2, figsize=(14,4))

# Plot clean_burn_cwc_difference on the first axis
clean_burn_cwc_difference.plot(ax=axes[0],
                               cmap='RdBu',
                               center=0,
                               cbar_kwargs={'label':'CWC Difference (g/cm²)'}
                              )
axes[0].set_title('Canopy Water Content (CWC) Difference Burned Tile')
axes[0].set_ylabel('Latitude')
axes[0].set_xlabel('Longitude')

# Plot clean_unburn_cwc_difference on the second axis
clean_unburn_cwc_difference.plot(ax=axes[1],
                                 cmap='RdBu',
                                 center=0,
                                 cbar_kwargs={'label':'CWC Difference (g/cm²)'}
                                )
axes[1].set_title('Canopy Water Content (CWC) Difference Unburned Tile')
axes[1].set_ylabel('Latitude')
axes[1].set_xlabel('Longitude')

plt.show()

#### Finding the difference in CWC measurements derived by NEON reflectance data and CWC measurements derived by EMIT reflectance data shows that EMIT CWC measurements are overall lower than NEON CWC measurements.

Remember that we created these plots by subtracting the resampled NEON dataset from the the EMIT dataset for the burned and unburned tiles. Visually, we can see a majority of values in the plots above are red, indicating that EMIT CWC values are consistently lower than NEON CWC values. This is confirmed by the mean unburned and burned CWC differences found in the cell above:

* The mean unburned Canopy Water Content (CWC) difference is: -0.0805 g/cm²
* The mean burned Canopy Water Content (CWC) difference is: -0.0986 g/cm²

EMIT CWC calculations being lower than NEON CWC calculations is confirmed even further by the histogram made in the cell below. The histogram shows that a majority of the difference values are negative, which means EMIT CWC calculations are lower than NEON.

The difference plots, mean values, and histogram confirm what we were noticing in the first CWC plots above - that EMIT CWC is lower overall than NEON CWC. Remember that the EMIT CWC values could be lower because the EMIT reflectance data used to calculate CWC is from July 2023 while NEON CWC is based on 2024 reflectance measurements. It could also be due to resolution differences or other factors such as different atmospheric corrections applied to the two datasets.

This difference is very important when thinking about forest health. If just EMIT is used to evaluate forest health, it may make a forest seem less healthy than it is. NEON would also need to be used to get a more accurate picture of CWC and forest health. Also, when using NEON and EMIT CWC calculations together, a correction factor would need to be applied.

In [ ]:
# Create a figure and axes for the histogram
fig, ax = plt.subplots(figsize=(8, 4))

# Plot the histogram.
# We use .values to access the underlying numpy array, and .flatten()
# to ensure the 2D raster data is treated as a 1D array of values.
# The 'bins' parameter controls the number of bars in the histogram.
ax.hist(clean_burn_cwc_difference
        .values
        .flatten(),
        bins=50,
        edgecolor='black',
        alpha=0.7)

# Add a vertical line at zero to show the neutral difference
ax.axvline(0,
           color='red',
           linestyle='dashed',
           linewidth=2,
           label='Zero Difference')

# Add a title and axis labels
ax.set_title('Frequency Distribution of CWC Difference in Burned Area')
ax.set_xlabel('CWC Difference (g/cm²)')
ax.set_ylabel('Frequency')

# Add a legend to explain the vertical line
ax.legend()

# Display the plot
plt.show()

#### Next Steps:

If you are interested in exploring this topic further, here are some possible next steps:
* CWC values could be classified into forest health categories such as "healthy", "struggling", and "dead".
* CWC values from different seasons could be calculated: NEON reflectance data is gathered during summer months (near peak phenological greenness) while EMIT reflectance data is gathered throughout the year so EMIT reflectance data could provide opportunities to assess seasonal patterns.
* CWC could be modeled to extrapolate forest health from NEON sites to a continental or global scale.